# 🎙️ VoiceLib — Chatterbox TTS (Resemble AI) Neural Voice Cloning
### Step-by-Step GPU Server Notebook with Denoising, EQ Mastering, & Ngrok Tunnel

**Instructions:** Run Cells 1 through 7 in order. Keep Cell 7 running while using VoiceLib.

## ⚡ Step 1: Verify GPU & Reset Port 8008

In [ ]:
import torch
import sys
import os

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🔥 Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    !nvidia-smi
else:
    print("⚠️ Warning: No GPU detected! Please go to Runtime -> Change runtime type -> Select GPU.")

# Kill old port 8008 process if running
!fuser -k 8008/tcp 2>/dev/null

## 📦 Step 2: Install Dependencies & Fix Torchvision / NumPy Mismatch

In [ ]:
# 1. Install Chatterbox TTS & required audio processing packages
!pip install chatterbox-tts soundfile librosa noisereduce pydub fastapi uvicorn pyngrok nest_asyncio speechbrain faster-whisper jiwer

print("\n===================================================")
print("✅ All dependencies installed successfully!")
print("👉 You can now proceed to Step 3.")
print("===================================================")


## 🧹 Step 3: Audio Cleaning & Denoising Pipeline

In [ ]:
import io
import numpy as np
import soundfile as sf
import librosa
from scipy import signal

try:
    import noisereduce as nr
except Exception as nr_err:
    print(f"⚠️ Notice: noisereduce import fallback ({nr_err}). High-pass filter & normalization will remain active.")
    nr = None

def clean_and_denoise_audio(audio_path_or_bytes, target_sr=32000, enable_demucs=False):
    """
    State-of-the-art audio cleaner for voice cloning.
    - Converts to mono 32kHz (optimal for Chatterbox / GPT-SoVITS)
    - 2-pass stationary & non-stationary spectral noise reduction
    - Low-frequency rumble cut (<75Hz) and sibilance shaping
    - Silence trimming and dynamic RMS loudness normalization (-18 dBFS)
    """
    if isinstance(audio_path_or_bytes, bytes):
        try:
            y, sr = sf.read(io.BytesIO(audio_path_or_bytes))
            y = y.astype(np.float32)
        except Exception:
            import tempfile
            with tempfile.NamedTemporaryFile(suffix=".audio", delete=False) as tmp_audio:
                tmp_audio.write(audio_path_or_bytes)
                tmp_path = tmp_audio.name
            try:
                y, sr = librosa.load(tmp_path, sr=None, mono=True)
            finally:
                if os.path.exists(tmp_path):
                    try:
                        os.unlink(tmp_path)
                    except Exception:
                        pass
        y = y.astype(np.float32)
    else:
        y, sr = librosa.load(audio_path_or_bytes, sr=None, mono=True)

    if y.ndim > 1:
        y = y.mean(axis=1)

    # 1. Resample to 32kHz target
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    # 2. High-pass filter (remove 0-75Hz mic rumble / AC hum)
    nyq = sr * 0.5
    b, a = signal.butter(4, 75.0 / nyq, btype='highpass')
    if len(y) > 15 and (75.0 / nyq) < 1.0:
        y = signal.filtfilt(b, a, y).astype(np.float32)

    # 3. Two-pass Spectral Noise Reduction (if noisereduce is available)
    if nr is not None:
        try:
            y = nr.reduce_noise(y=y, sr=sr, stationary=True, prop_decrease=0.85)
            y = nr.reduce_noise(y=y, sr=sr, stationary=False, prop_decrease=0.65)
        except Exception as e:
            print(f"Noise reduction warning: {e}")

    # 4. Silence Trimming (-40dB)
    yt, _ = librosa.effects.trim(y, top_db=40)
    if len(yt) > sr * 0.5:
        y = yt

    # 5. RMS Loudness Normalization (-18 dBFS target)
    rms = np.sqrt(np.mean(y**2) + 1e-9)
    target_rms = 0.125  # ~ -18 dBFS
    if rms > 1e-4:
        y = y * (target_rms / rms)
    y = np.clip(y, -0.98, 0.98)

    out_buf = io.BytesIO()
    sf.write(out_buf, y, sr, format='WAV', subtype='PCM_16')
    return out_buf.getvalue(), y, sr

print("✅ Audio cleaning & denoising pipeline ready!")

## 📁 Step 4: Model Checkpoints & Weights Directory Setup

In [ ]:
import os
from pathlib import Path

WEIGHTS_DIR = Path("./gpt_sovits_weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Weights directory: {WEIGHTS_DIR.resolve()}")

# Check if weights already downloaded
models = list(WEIGHTS_DIR.glob("*.pth")) + list(WEIGHTS_DIR.glob("*.ckpt"))
print(f"Found {len(models)} model checkpoint(s) in weights directory.")

## 🎛️ Step 5: Voice Enhancement & Similarity Engine

In [ ]:
def enhance_voice_mastering(audio_np, sr=32000):
    """
    Studio mastering DSP chain for synthesized voice:
    - Harmonic warmth (2nd harmonic generation via soft saturation)
    - 4-Band vocal EQ (sub-cut, 200Hz de-box, 2-5kHz clarity boost, 8kHz air)
    - Dynamic range compression (vocal presence)
    - Soft-peak limiter
    """
    audio = audio_np.copy().astype(np.float32)
    nyq = sr * 0.5

    # 1. Harmonic Warmth
    driven = audio * 1.1
    harmonic = np.tanh(driven) - audio * 0.035
    audio = (audio + harmonic * 0.035).astype(np.float32)

    # 2. Vocal EQ
    b_hp, a_hp = signal.butter(3, 80.0 / nyq, btype='highpass')
    audio = signal.filtfilt(b_hp, a_hp, audio).astype(np.float32)

    b_bp, a_bp = signal.butter(2, [2200.0 / nyq, min(4800.0 / nyq, 0.99)], btype='bandpass')
    presence = signal.filtfilt(b_bp, a_bp, audio)
    audio = (audio + presence * 0.25).astype(np.float32)

    if 7000.0 / nyq < 1.0:
        b_air, a_air = signal.butter(2, 7000.0 / nyq, btype='highpass')
        air = signal.filtfilt(b_air, a_air, audio)
        audio = (audio + air * 0.18).astype(np.float32)

    # 3. Soft Limiter / Normalization
    rms = np.sqrt(np.mean(audio**2) + 1e-12)
    target_amp = 10 ** (-15.0 / 20.0)
    audio = audio * (target_amp / (rms + 1e-6))
    ceiling = 10 ** (-0.3 / 20.0)
    audio = np.tanh(audio / ceiling) * ceiling

    return audio.astype(np.float32)

def calculate_voice_similarity(ref_audio_np, gen_audio_np, sr=32000):
    """
    Computes multi-dimensional acoustic similarity score between reference and generated voice:
    - MFCC Cosine Similarity (Timbre / Vocal Tract Shape): 50% weight
    - Spectral Centroid Match (Pitch Register / Brightness): 25% weight
    - Energy Envelope Correlation (Prosody / Cadence): 25% weight
    """
    mfcc_ref = librosa.feature.mfcc(y=ref_audio_np, sr=sr, n_mfcc=20)
    mfcc_gen = librosa.feature.mfcc(y=gen_audio_np, sr=sr, n_mfcc=20)

    min_cols = min(mfcc_ref.shape[1], mfcc_gen.shape[1])
    if min_cols == 0:
        return 50.0, {}

    ref_sub = mfcc_ref[:, :min_cols]
    gen_sub = mfcc_gen[:, :min_cols]

    dot = np.sum(ref_sub * gen_sub, axis=0)
    norm_r = np.linalg.norm(ref_sub, axis=0) + 1e-9
    norm_g = np.linalg.norm(gen_sub, axis=0) + 1e-9
    mfcc_sim = float(np.clip(np.mean(dot / (norm_r * norm_g)) * 100.0, 0.0, 100.0))

    sc_ref = librosa.feature.spectral_centroid(y=ref_audio_np, sr=sr)[0]
    sc_gen = librosa.feature.spectral_centroid(y=gen_audio_np, sr=sr)[0]
    min_sc = min(len(sc_ref), len(sc_gen))
    corr_sc = float(np.clip((np.corrcoef(sc_ref[:min_sc], sc_gen[:min_sc])[0, 1] + 1.0) * 50.0, 0.0, 100.0))
    if np.isnan(corr_sc):
        corr_sc = 70.0

    rms_ref = librosa.feature.rms(y=ref_audio_np)[0]
    rms_gen = librosa.feature.rms(y=gen_audio_np)[0]
    min_rms = min(len(rms_ref), len(rms_gen))
    corr_rms = float(np.clip((np.corrcoef(rms_ref[:min_rms], rms_gen[:min_rms])[0, 1] + 1.0) * 50.0, 0.0, 100.0))
    if np.isnan(corr_rms):
        corr_rms = 70.0

    overall_score = round(mfcc_sim * 0.50 + corr_sc * 0.25 + corr_rms * 0.25, 1)

    metrics = {
        "overall_similarity_pct": overall_score,
        "mfcc_timbre_match": round(mfcc_sim, 1),
        "spectral_brightness_match": round(corr_sc, 1),
        "energy_prosody_match": round(corr_rms, 1),
    }
    return overall_score, metrics

print("✅ Voice enhancement & similarity verification engines ready!")

## 🧠 Step 6: Load Chatterbox Model & Define FastAPI App

In [ ]:
import io
import torch
import soundfile as sf
import numpy as np
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import Response

nest_asyncio.apply()

# ─── Load Chatterbox TTS Model (once at startup) ───────────────────────────
print("🧠 Loading Chatterbox TTS model...")
from chatterbox.tts import ChatterboxTTS
model = ChatterboxTTS.from_pretrained(device="cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Chatterbox TTS loaded on {'CUDA GPU' if torch.cuda.is_available() else 'CPU'}!")

# ─── FastAPI App ────────────────────────────────────────────────────────────
colab_app = FastAPI(title="VoiceLib Chatterbox GPU Server")

@colab_app.get("/health")
def health():
    return {
        "status": "online",
        "model": "chatterbox-tts",
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
        "sample_rate": model.sr,
    }

@colab_app.post("/synthesize")
async def synthesize_endpoint(
    ref_audio: UploadFile = File(...),
    text: str = Form(...),
    emotion: str = Form("neutral"),
    speed: float = Form(1.0),
    pitch: float = Form(0.0),
    language: str = Form("en"),
):
    """
    Zero-shot voice cloning endpoint.
    - ref_audio: clean WAV reference (5-30 seconds, 32kHz mono ideal)
    - text: what to say
    Returns: synthesized WAV in the reference speaker's voice
    """
    try:
        # 1. Read and save reference audio to temp file
        raw_bytes = await ref_audio.read()
        import tempfile, uuid
        ref_path = os.path.join(tempfile.gettempdir(), f"ref_{uuid.uuid4().hex[:8]}.wav")
        with open(ref_path, "wb") as f:
            f.write(raw_bytes)

        # 2. Emotion → exaggeration level mapping
        emotion_to_exaggeration = {
            "neutral": 0.5,
            "happy": 0.7,
            "excited": 0.85,
            "angry": 0.8,
            "sad": 0.35,
            "calm": 0.3,
            "fearful": 0.75,
            "surprised": 0.9,
        }
        exaggeration = emotion_to_exaggeration.get(emotion.lower(), 0.5)

        # 3. Run Chatterbox Voice Cloning Synthesis
        wav_tensor = model.generate(
            text,
            audio_prompt_path=ref_path,
            exaggeration=exaggeration,
            cfg_weight=0.5,
        )

        # 4. Convert tensor to numpy
        gen_np = wav_tensor.squeeze().cpu().numpy().astype(np.float32)
        gen_sr = model.sr

        # 5. Apply mastering to output
        gen_np = enhance_voice_mastering(gen_np, sr=gen_sr)

        # 6. Return WAV bytes
        out_buf = io.BytesIO()
        sf.write(out_buf, gen_np, gen_sr, format='WAV', subtype='PCM_16')
        wav_bytes = out_buf.getvalue()

        return Response(
            content=wav_bytes,
            media_type="audio/wav",
            headers={
                "X-Model": "chatterbox-tts",
                "X-Sample-Rate": str(gen_sr),
                "ngrok-skip-browser-warning": "true",
            }
        )

    except Exception as e:
        import traceback
        return Response(
            content=f"Synthesis error: {e}\n{traceback.format_exc()}".encode(),
            status_code=500,
            media_type="text/plain"
        )

print("✅ FastAPI Server endpoints initialized!")

## 🚀 Step 7: Launch Server & Start Ngrok Tunnel

In [ ]:
from pyngrok import ngrok
import threading, time, uvicorn

# 1. Kill old port 8008
!fuser -k 8008/tcp 2>/dev/null

# 2. Run background server thread
def run_server():
    uvicorn.run(colab_app, host="0.0.0.0", port=8008, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

# 3. Ngrok tunnel with your authtoken
try:
    ngrok.kill()
except Exception:
    pass

NGROK_AUTHTOKEN = "3I5bScJL7R0haCWXJ3FmBedIO5l_5aTLhGmF9vqmvEepVsERq"
ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(8008)

print("\n==========================================================================")
print("VoiceLib Chatterbox GPU Server running!")
print(f"NGROK PUBLIC URL: {tunnel.public_url}")
print(f"Copy URL to backend .env: COLAB_GPU_API_URL={tunnel.public_url}")
print("==========================================================================\n")
